In [1]:

import pandas as pd
clh = pd.read_csv('Customer Loyalty History.csv')
cfa = pd.read_csv('Customer Flight Activity.csv')


print(clh.shape)
print(cfa.shape)
print(clh.columns.tolist())
print(cfa.columns.tolist())

(16737, 16)
(392936, 8)
['Loyalty Number', 'Country', 'Province', 'City', 'Postal Code', 'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card', 'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month', 'Cancellation Year', 'Cancellation Month']
['Loyalty Number', 'Year', 'Month', 'Total Flights', 'Distance', 'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed']


In [2]:
# Check for missing value
print(clh.isnull().sum())
print()
print(cfa.isnull().sum())

# Check cancellationss
print("Cancelled members:",clh['Cancellation Year'].notna().sum())
print("Active members:",clh['Cancellation Year'].isna().sum())

# Check year range in flights
print(cfa['Year'].value_counts().sort_index())

Loyalty Number            0
Country                   0
Province                  0
City                      0
Postal Code               0
Gender                    0
Education                 0
Salary                 4238
Marital Status            0
Loyalty Card              0
CLV                       0
Enrollment Type           0
Enrollment Year           0
Enrollment Month          0
Cancellation Year     14670
Cancellation Month    14670
dtype: int64

Loyalty Number                 0
Year                           0
Month                          0
Total Flights                  0
Distance                       0
Points Accumulated             0
Points Redeemed                0
Dollar Cost Points Redeemed    0
dtype: int64
Cancelled members: 2067
Active members: 14670
Year
2017    191100
2018    201836
Name: count, dtype: int64


In [ ]:
# Fill missing salary with median
clh['Salary'] = clh['Salary'].fillna(clh['Salary'].median())

# Check for duplicate members
print("Duplicate members in clh:", clh['Loyalty Number'].duplicated().sum())

# Check for duplicate month entries in cfa
# (same member, same year, same month appearing twice)
dupes = cfa.duplicated(subset=['Loyalty Number', 'Year', 'Month'])
print("Duplicate month rows in cfa:", dupes.sum())

# Check for weird values — negative flights or distance?
print("\n", cfa['Total Flights'].min())
print( cfa['Distance'].min())
print("Min points accumulated:", cfa['Points Accumulated'].min())

Duplicate members in clh: 0
Duplicate month rows in cfa: 3871

Min flights: 0
Min distance: 0
Min points accumulated: 0.0


In [5]:
# Fix duplicate month rows by aggregating (summing) them
cfa_clean = cfa.groupby(
    ['Loyalty Number', 'Year', 'Month'],
    as_index=False
).agg({
    'Total Flights'              : 'sum',
    'Distance'                   : 'sum',
    'Points Accumulated'         : 'sum',
    'Points Redeemed'            : 'sum',
    'Dollar Cost Points Redeemed': 'sum'
})

print("Original cfa rows:", len(cfa))
print("Cleaned cfa rows: ", len(cfa_clean))
print("Duplicates remaining:", cfa_clean.duplicated(
    subset=['Loyalty Number','Year','Month']).sum())

Original cfa rows: 392936
Cleaned cfa rows:  389065
Duplicates remaining: 0


In [6]:
# Step 1: Who formally cancelled?
cancelled_ids = set(clh[clh['Cancellation Year'].notna()]['Loyalty Number'])
print("Formally cancelled:", len(cancelled_ids))

# Step 2: Who flew zero flights in 2018?
flights_2018 = cfa_clean[cfa_clean['Year'] == 2018].groupby(
    'Loyalty Number')['Total Flights'].sum()

zero_in_2018 = set(flights_2018[flights_2018 == 0].index)

# Also include active members who don't even appear in 2018 data
appeared_in_2018 = set(cfa_clean[cfa_clean['Year'] == 2018]['Loyalty Number'])
never_appeared   = set(clh['Loyalty Number']) - appeared_in_2018
silent_2018      = zero_in_2018 | never_appeared

# Remove formally cancelled (already counted)
behavioral_churners = silent_2018 - cancelled_ids
print("Behavioral churners (active but silent in 2018):", len(behavioral_churners))

# Step 3: Combined churn label
clh['churn'] = clh['Loyalty Number'].apply(
    lambda x: 1 if x in cancelled_ids or x in behavioral_churners else 0
)

print("\nTotal churned:", clh['churn'].sum())
print("Total active:", (clh['churn'] == 0).sum())
print("Churn rate:", round(clh['churn'].mean() * 100, 1), "%")

Formally cancelled: 2067
Behavioral churners (active but silent in 2018): 661

Total churned: 2728
Total active: 14009
Churn rate: 16.3 %


In [7]:
# Only use 2017 flight data for features
cfa_2017 = cfa_clean[cfa_clean['Year'] == 2017]

features = cfa_2017.groupby('Loyalty Number').agg(
    total_flights     = ('Total Flights',               'sum'),
    total_distance    = ('Distance',                    'sum'),
    total_pts_acc     = ('Points Accumulated',          'sum'),
    total_pts_red     = ('Points Redeemed',             'sum'),
    avg_flights_month = ('Total Flights',               'mean'),
    flight_std        = ('Total Flights',               'std'),
    active_months     = ('Month',                       'count'),
).reset_index()

# Flight consistency — members who fly irregularly churn more
# High std = erratic flyer, so we invert it
features['flight_consistency'] = 1 / (features['flight_std'] + 1)

# Redemption ratio — are they actively using their points?
# Low ratio = passive accumulator = less engaged
features['redemption_ratio'] = (
    features['total_pts_red'] / (features['total_pts_acc'] + 1)
)

# Tenure — how long have they been a member (as of 2018)?
clh['tenure_years'] = (
    2018 - clh['Enrollment Year'] - clh['Enrollment Month'] / 12
).clip(lower=0)

print("Features shape:", features.shape)
print(features.head(3))

Features shape: (15766, 10)
   Loyalty Number  total_flights  total_distance  total_pts_acc  \
0          100018             24           37760        37760.0   
1          100102             25           39884        39884.0   
2          100140             22           31582        31582.0   

   total_pts_red  avg_flights_month  flight_std  active_months  \
0           1128           2.000000    1.414214             12   
1           1195           2.083333    1.781640             12   
2              0           1.833333    1.696699             12   

   flight_consistency  redemption_ratio  
0            0.414214          0.029872  
1            0.359500          0.029961  
2            0.370824          0.000000  


In [ ]:
# Merge features with member profile
df = clh.merge(features, on='Loyalty Number', how='left')

# Members with no 2017 flight data get 0s (they were completely inactive)
flight_cols = ['total_flights', 'total_distance', 'total_pts_acc',
               'total_pts_red', 'avg_flights_month', 'flight_std',
               'active_months', 'flight_consistency', 'redemption_ratio']
df[flight_cols] = df[flight_cols].fillna(0)

# from week 2 SA
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in ['Gender', 'Education', 'Marital Status', 
            'Loyalty Card', 'Enrollment Type']:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

print("Final dataframe shape:", df.shape)
print("Churn distribution:\n", df['churn'].value_counts())

Final dataframe shape: (16737, 32)
Churn distribution:
 churn
0    14009
1     2728
Name: count, dtype: int64


In [10]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

# Define which columns are features
feature_cols = [
    'total_flights', 'total_distance', 'total_pts_acc', 'total_pts_red',
    'avg_flights_month', 'flight_consistency', 'redemption_ratio',
    'active_months', 'tenure_years', 'Salary', 'CLV',
    'Gender_enc', 'Education_enc', 'Marital Status_enc',
    'Loyalty Card_enc', 'Enrollment Type_enc'
]

X = df[feature_cols]
y = df['churn']

# Split: 80% train, 20% test
# stratify=y ensures both splits have same churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Model 1: Logistic Regression (simple baseline)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr_s, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_te_s)[:,1])

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, 
                             class_weight='balanced')
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

# Model 3: Gradient Boosting (usually best for tabular data)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, 
                                 random_state=42)
gb.fit(X_train, y_train)
gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:,1])

print("=== MODEL COMPARISON ===")
print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Random Forest AUC:       {rf_auc:.4f}")
print(f"Gradient Boosting AUC:   {gb_auc:.4f}")

=== MODEL COMPARISON ===
Logistic Regression AUC: 0.8136
Random Forest AUC:       0.8221
Gradient Boosting AUC:   0.8515


In [11]:
import pandas as pd

# Feature importance — what did the model actually learn?
fi = pd.Series(
    gb.feature_importances_, 
    index=feature_cols
).sort_values(ascending=False)

print("=== TOP FEATURES ===")
print(fi.head(8).round(4))

# Score every member with their churn probability
df['churn_prob'] = gb.predict_proba(X[feature_cols])[:,1]

# Assign risk tier
df['risk_tier'] = pd.cut(
    df['churn_prob'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print("\n=== RISK TIER DISTRIBUTION ===")
print(df['risk_tier'].value_counts())

=== TOP FEATURES ===
tenure_years          0.4982
flight_consistency    0.4049
total_flights         0.0311
avg_flights_month     0.0239
CLV                   0.0107
total_pts_acc         0.0074
total_distance        0.0072
Salary                0.0062
dtype: float64

=== RISK TIER DISTRIBUTION ===
risk_tier
Low Risk       15182
High Risk       1314
Medium Risk      241
Name: count, dtype: int64


In [12]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Features that describe customer VALUE
seg_features = ['CLV', 'total_flights', 'tenure_years', 
                'avg_flights_month', 'redemption_ratio', 'Salary']

X_seg = df[seg_features].fillna(0)
scaler2 = StandardScaler()
X_seg_scaled = scaler2.fit_transform(X_seg)

# KMeans with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(X_seg_scaled)

# Label each cluster by its average CLV rank
seg_profile = df.groupby('segment').agg(
    count        = ('Loyalty Number', 'count'),
    avg_clv      = ('CLV', 'mean'),
    avg_flights  = ('total_flights', 'mean'),
    avg_tenure   = ('tenure_years', 'mean'),
    churn_rate   = ('churn', 'mean'),
    avg_salary   = ('Salary', 'mean')
).round(2)

print(seg_profile.sort_values('avg_clv', ascending=False))

         count  avg_clv  avg_flights  avg_tenure  churn_rate  avg_salary
segment                                                                 
0         6303  8135.00         2.46        0.96        0.35    72695.40
1         9969  7913.56        20.19        3.11        0.05    74306.16
3            8  7855.42         0.00        0.80        0.25    70142.62
2          457  7619.63        14.21        2.42        0.16   223795.98


In [13]:
# Map segment numbers to names
seg_names = {0: 'Champions', 1: 'Loyalists', 2: 'Dormant', 3: 'At-Risk'}
df['segment_label'] = df['segment'].map(seg_names)

# Save the final scored dataset
df.to_csv('final_results.csv', index=False)
print("Saved! Columns:", df.columns.tolist())
print("\nSegment distribution:")
print(df['segment_label'].value_counts())

Saved! Columns: ['Loyalty Number', 'Country', 'Province', 'City', 'Postal Code', 'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card', 'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month', 'Cancellation Year', 'Cancellation Month', 'churn', 'tenure_years', 'total_flights', 'total_distance', 'total_pts_acc', 'total_pts_red', 'avg_flights_month', 'flight_std', 'active_months', 'flight_consistency', 'redemption_ratio', 'Gender_enc', 'Education_enc', 'Marital Status_enc', 'Loyalty Card_enc', 'Enrollment Type_enc', 'churn_prob', 'risk_tier', 'segment', 'segment_label']

Segment distribution:
segment_label
Loyalists    9969
Champions    6303
Dormant       457
At-Risk         8
Name: count, dtype: int64
